# Explore NWB structure

Raw `pynwb` extraction of the datasets relevant to the WM slots/gain reanalysis, so we can
see their structure before committing to a pipeline. No decoding, no binning here.

Change `SESSION` and rerun. Named targets: DMFC = `2022-06-01`, FEF = `2022-06-04`
(both in the 3-position range Perle 05-26..06-05). Each session file carries both probes
(Neuropixel `s0` = DMFC right hemi; V-probe `vprobe*` = FEF left hemi).

Kernel: project `.venv` (uv).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pynwb

SESSION = "2022-06-01"
ROOT = Path.cwd().parent
behav_path = ROOT / f"data/Perle/behav/sub-Perle_ses-{SESSION}_behavior+task.nwb"
spikes_path = ROOT / f"data/Perle/ephys/spikes/sub-Perle_ses-{SESSION}_spikesorting.nwb"
behav_path.exists(), spikes_path.exists()

## Behavior + task file

In [ ]:
behav = pynwb.NWBHDF5IO(str(behav_path), mode="r").read()
print(behav.session_description[:200])
print("intervals:", list(behav.intervals))
print("processing:", list(behav.processing))
behav

In [ ]:
# trials table -> DataFrame. Index is the NWB trial id.
trials = behav.intervals["trials"].to_dataframe()
print(trials.shape)
print(list(trials.columns))
trials.head()

In [ ]:
# The per-object columns are JSON-serialized lists (one element per object on screen).
# JSON, not python literal: booleans are lowercase true/false, so use json.loads.
obj_cols = ["stimulus_object_identities", "stimulus_object_positions",
            "stimulus_object_target", "stimulus_object_velocities"]
parsed = {c: trials[c].map(json.loads) for c in obj_cols}

n_obj = parsed["stimulus_object_identities"].map(len)
print("objects per trial:")
print(n_obj.value_counts().sort_index())

# One example trial, fully unpacked.
i = trials.index[0]
for c in obj_cols:
    print(c, "->", parsed[c].loc[i])

In [ ]:
# Single-object trials are the analysis substrate. Look at identity/position structure.
single = n_obj == 1
print("single-object trials:", int(single.sum()), "of", len(trials))
print("broke_fixation among single-object:", int(trials.loc[single, "broke_fixation"].sum()))

ids = parsed["stimulus_object_identities"][single].map(lambda l: l[0])
xy = np.array([p[0] for p in parsed["stimulus_object_positions"][single]])
print("identities:", pd.Series(ids).value_counts().to_dict())
print("unique positions (rounded):")
print(np.unique(xy.round(3), axis=0))

In [ ]:
# Phase-onset timing columns define the trial windows (stimulus 1.0s, delay 1.0s).
phase_cols = [c for c in trials.columns if c.startswith("phase_")]
print(phase_cols)
trials.loc[single, phase_cols].head()

## Spike sorting file

In [ ]:
spikes = pynwb.NWBHDF5IO(str(spikes_path), mode="r").read()
units = spikes.processing["ecephys"]["units"]
n_units = len(units.id)
print("units:", n_units)
print("colnames:", units.colnames)

In [ ]:
# Per-unit metadata: quality, depth, and which probe/group each unit sits on.
group = np.asarray(units["electrodes_group"][:])
quality = np.asarray(units["quality"][:])
print("probe group counts:", pd.Series(group).value_counts().to_dict())
print("quality counts:", pd.Series(quality).value_counts().to_dict())

# electrode_group descriptions carry the stereotactic entry coords used to infer region.
for name, g in spikes.electrode_groups.items():
    print(f"\n[{name}]", g.description[:300])

In [ ]:
# spike_times: one ragged array per unit.
st0 = np.asarray(units["spike_times"][0])
print("unit 0 spike_times:", st0.shape, st0[:5], "...")
print("session span (s):", st0.min(), "->", np.asarray(units["spike_times"][n_units - 1]).max())

In [ ]:
# obs_trials: per-unit boolean mask over trial ids. Zero counts outside the mask are
# missing, not silence. Shape should be [n_units x n_trials].
obs = np.stack([np.asarray(x) for x in units["obs_trials"][:]])
print("obs shape:", obs.shape)
coverage = obs.mean(axis=1)
print("per-unit trial coverage: median", round(float(np.median(coverage)), 3),
      "| fully-observed units:", int((coverage == 1).sum()), "of", n_units)